# 02 — Preprocessing & Feature Engineering

Tiền xử lý dữ liệu và xây dựng đặc trưng:
- Làm sạch dữ liệu (missing, outlier, duplicate)
- Thêm đặc trưng thời gian và shipping days
- Xây dựng đặc trưng RFM cho clustering
- Tổng hợp doanh số theo tháng cho forecasting

In [ ]:
# %% [import] Thư viện và cấu hình
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_params, load_raw_data, get_data_info
from src.data.cleaner import (
    drop_unnecessary_columns, drop_duplicates, handle_missing,
    remove_outliers_sales, add_time_features, add_shipping_days,
    run_cleaning_pipeline,
)
from src.features import build_customer_features, scale_features, build_monthly_sales

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.2f}".format)


In [ ]:
# %% [load] Đọc dữ liệu thô
params = load_params()
df_raw = load_raw_data(params)
print(f"Raw shape: {df_raw.shape}")
df_raw.head()

## 1. Kiểm tra trước tiền xử lý

In [ ]:
# %% [check] Thống kê missing, duplicate, kiểu dữ liệu trước xử lý
info_before = get_data_info(df_raw)
print("=== TRƯỚC TIỀN XỬ LÝ ===")
print(f"Shape      : {info_before['shape']}")
print(f"Duplicates : {info_before['duplicates']}")
print(f"Missing    :")
for col, n in info_before['missing'].items():
    if n > 0:
        print(f"  {col}: {n}")
print("\nDtypes:")
print(df_raw.dtypes)

In [ ]:
# %% [check] Phân phối Sales trước khi xử lý outlier
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df_raw["Sales"], kde=True, ax=axes[0], color="coral")
axes[0].set_title("Sales (trước xử lý)")
sns.boxplot(x=df_raw["Sales"], ax=axes[1], color="coral")
axes[1].set_title("Boxplot Sales (trước xử lý)")
plt.tight_layout()
plt.show()

## 2. Thực hiện tiền xử lý

In [ ]:
# %% [clean] Xoá cột không cần thiết theo params
df = drop_unnecessary_columns(df_raw, params["preprocessing"]["drop_columns"])
print(f"Sau drop columns: {df.shape} — Đã xoá: {params['preprocessing']['drop_columns']}")

In [ ]:
# %% [clean] Xoá dòng trùng lặp
df = drop_duplicates(df)

In [ ]:
# %% [clean] Xử lý giá trị thiếu
df = handle_missing(df)
print(f"Missing sau xử lý: {df.isnull().sum().sum()}")

In [ ]:
# %% [clean] Loại bỏ outlier Sales bằng Z-score (ngưỡng 3.0)
df = remove_outliers_sales(df)

In [ ]:
# %% [feature] Thêm đặc trưng thời gian: Year, Month, Quarter, DayOfWeek
df = add_time_features(df)
print(df[["Order Date", "Year", "Month", "Quarter", "DayOfWeek"]].head())

In [ ]:
# %% [feature] Thêm Shipping Days = Ship Date - Order Date
df = add_shipping_days(df)
print(df[["Order Date", "Ship Date", "Shipping Days"]].head())

## 3. Kiểm tra sau tiền xử lý

In [ ]:
# %% [check] So sánh shape và phân phối Sales trước - sau xử lý
print("=== SAU TIỀN XỬ LÝ ===")
print(f"Shape      : {df.shape}")
print(f"Duplicates : {df.duplicated().sum()}")
print(f"Missing    : {df.isnull().sum().sum()}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df["Sales"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Sales (sau xử lý)")
sns.boxplot(x=df["Sales"], ax=axes[1], color="steelblue")
axes[1].set_title("Boxplot Sales (sau xử lý)")
plt.tight_layout()
plt.savefig("../outputs/figures/sales_before_after.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# %% [save] Lưu dữ liệu đã xử lý ra data/processed/
from src.data.cleaner import save_processed
save_processed(df, params)

## 4. Xây dựng đặc trưng RFM

In [ ]:
# %% [feature] Tính RFM (Recency, Frequency, Monetary) theo Customer ID
from src.features.builder import build_rfm, build_avg_order_value, build_category_diversity

rfm       = build_rfm(df, params)
aov       = build_avg_order_value(df)
diversity = build_category_diversity(df)

print(f"RFM shape: {rfm.shape}")
rfm.describe()

In [ ]:
# %% [plot] Phân phối 3 chỉ số RFM
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes, ["Recency", "Frequency", "Monetary"],
                                 ["coral", "steelblue", "green"]):
    sns.histplot(rfm[col], kde=True, ax=ax, color=color)
    ax.set_title(f"Phân phối {col}")
plt.tight_layout()
plt.savefig("../outputs/figures/rfm_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# %% [feature] Gộp toàn bộ đặc trưng khách hàng và chuẩn hoá
customer_features       = build_customer_features(df, params)
customer_scaled, scaler = scale_features(customer_features)

print(f"Customer features shape: {customer_features.shape}")
print(f"Columns: {list(customer_features.columns)}")
customer_features.head()

## 5. Xây dựng chuỗi thời gian doanh số theo tháng

In [ ]:
# %% [feature] Tổng hợp doanh số theo tháng cho forecasting
from src.features.builder import build_lag_features

monthly    = build_monthly_sales(df)
monthly_lag = build_lag_features(monthly)

print(f"Monthly sales shape: {monthly.shape}")
print(f"Với lag features   : {monthly_lag.shape}")
monthly.tail()

In [ ]:
# %% [plot] Line chart doanh số theo tháng sau tiền xử lý
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(monthly["YearMonth"], monthly["Sales"], marker="o", color="steelblue", linewidth=2)
ax.fill_between(monthly["YearMonth"], monthly["Sales"], alpha=0.15, color="steelblue")
ax.set_title("Doanh số theo tháng (sau tiền xử lý)")
ax.set_xlabel("Tháng")
ax.set_ylabel("Sales (USD)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../outputs/figures/monthly_sales_clean.png", dpi=150, bbox_inches="tight")
plt.show()